In [1]:
import pandas as pd 
import json

case_data = 'data/CasE-mutated-peptides.csv'
mhc_data = 'data/MHC-protein-sequences.json'
df_case = pd.read_csv(case_data)
with open(mhc_data, 'r') as f:
    mhc_data = json.load(f)


In [17]:
df_case.tail(3)

,Peptide ID,Peptide,HLA restriction,Peptide Length,anchor position,aaWT,residue number,aaMUT
104,CasE_55_PΩV1,NGKIQTVCH,B*08:01,9,PΩ,F,163,H
105,CasE_55_PΩV2,NGKIQTVCN,B*08:01,9,PΩ,F,163,N
106,CasE_55_PΩV3,NGKIQTVCW,B*08:01,9,PΩ,F,163,W


In [3]:
import os 

"""
data/
└── input/
    └── af/
        ├── CasE_55_PΩV1/
        │   ├── wt.json
        │   └── mut.json
        └── CasE_55_PΩV2/
            ├── wt.json
            └── mut.json

{
    "name": "MHC-A01:01_CasE-B07:02_b2mgbl",
    "modelSeeds": [1],
    "dialect": "alphafold3",
    "version": 1,
    "sequences": [
        {"protein": {"id": ["A"], "sequence": ""}},
        {"protein": {"id": ["B"], "sequence": ""}},
        {"protein": {"id": ["C"], "sequence": ""}}
    ]
}
"""
# print(mhc_data['HLA-A*01:01']['Protein sequence'][24:308])
af_input = 'data/input/af'
os.makedirs(af_input, exist_ok=True)
b2m_seq = mhc_data['Beta-2-microglobulin']['Protein sequence main (21-119)']
for i in range(len(df_case)):
    Peptide_ID,	Peptide, HLA_restriction, Peptide_Length, anchor_position, aaWT, residue_number, aaMUT = df_case.iloc[i]
    hla_key = [k for k in mhc_data.keys() if HLA_restriction in k]
    print(hla_key)
    if not hla_key:
        print(HLA_restriction)
    else:
        hla_key = hla_key[0]

    segment = mhc_data[hla_key]['UniProt']['location']['Extracellular']
    start, end = map(int, segment.split("-"))
    hla_seq = mhc_data[hla_key]['Protein sequence']
    hla_seq_extracellular = hla_seq[start-1:end]

    try: 
        int_anchor_pos = int(anchor_position[-1])
    except:
        int_anchor_pos = Peptide_Length
    WT_Peptide = Peptide[:int_anchor_pos-1] + aaWT + Peptide[int_anchor_pos:]

    wt_dict = {
        "name": "wt", 
        "modelSeeds": [1], 
        "dialect": "alphafold3", 
        "version": 1,
        "sequences": [
            {"protein": {"id": ["A"], "sequence": hla_seq_extracellular}}, # MHC
            {"protein": {"id": ["B"], "sequence": WT_Peptide}}, # Peptide
            {"protein": {"id": ["C"], "sequence": b2m_seq}} # Beta-2-microglobulin (B2M)
        ],
    }
    mut_dict = {
        "name": "mut", 
        "modelSeeds": [1], 
        "dialect": "alphafold3", 
        "version": 1,
        "sequences": [
            {"protein": {"id": ["A"], "sequence": hla_seq_extracellular}},
            {"protein": {"id": ["B"], "sequence": Peptide}},
            {"protein": {"id": ["C"], "sequence": b2m_seq}}
        ],
    }
    os.makedirs(f'{af_input}/{Peptide_ID}', exist_ok=True)

    with open(f'{af_input}/{Peptide_ID}/wt.json', 'w') as f:
        json.dump(wt_dict, f, indent=2)
    with open(f'{af_input}/{Peptide_ID}/mut.json', 'w') as f:
        json.dump(mut_dict, f, indent=2)
    print(f'{af_input}/{Peptide_ID}/wt.json')
    print(f'{af_input}/{Peptide_ID}/mut.json')
    if i == 4:
        break

['HLA-B*07:02']
data/input/af/CasE_14_P2V1/wt.json
data/input/af/CasE_14_P2V1/mut.json
['HLA-B*07:02']
data/input/af/CasE_14_P2V2/wt.json
data/input/af/CasE_14_P2V2/mut.json
['HLA-B*07:02']
data/input/af/CasE_14_P2V3/wt.json
data/input/af/CasE_14_P2V3/mut.json
['HLA-B*07:02']
data/input/af/CasE_14_P2V4/wt.json
data/input/af/CasE_14_P2V4/mut.json
['HLA-B*07:02']
data/input/af/CasE_14_P2V5/wt.json
data/input/af/CasE_14_P2V5/mut.json


In [3]:
mhc_data['HLA-B*07:02']['Protein sequence'][24:]

'GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRNTQIYKAQAQTDRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHDQYAYDGKDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGECVEWLRRYLENGKDKLERADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWEPSSQSTVPIVGIVAGLAVLAVVVIGAVVAAVMCRRKSSGGKGGSYSQAACSDSAQGSDVSLTA'

In [2]:
import MDAnalysis as mda
from MDAnalysis.analysis import align
from Bio import pairwise2
from Bio.Align import PairwiseAligner
import itertools


/home/yang_loci/miniconda3/envs/enm_workshop/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yang_loci/miniconda3/envs/enm_workshop/lib/python3.11/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [4]:
three2one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def superimpose(mobile, reference, out='superimposed.pdb'):
    """Align a mobile structure to a reference using matched protein CA atoms.
    Flow:
    1. Load the mobile and reference structures, then keep only protein CA atoms.
    2. For each segment/chain, convert residue names to a one-letter sequence.
    3. Align reference and mobile sequences chain by chain in the order they appear.
       If the protein has two chains, the first reference chain is paired with the
       first mobile chain, and the second reference chain is paired with the second.
    4. Concatenate the per-chain alignments, keep only matched aligned residues, and
       use those CA atoms to calculate the rotation/translation.
    5. Apply the transform to all atoms in the mobile structure and write it out.
    Logic note:
    This works well when the two structures have the same chain order and comparable
    chain composition. If chain order differs, zip() may pair the wrong chains.
    """
    mobile = mda.Universe(mobile)
    reference = mda.Universe(reference)

    # Select protein and CA atoms 
    ref_ca = reference.select_atoms('protein and name CA').copy()
    mob_ca = mobile.select_atoms('protein and name CA').copy()

    # Extract sequences
    ref_seqs = []
    for segment in ref_ca.segments:
        chain_id = segment.segid 
        if chain_id == 'C':
            continue
        seq = ''.join([three2one[r.resname]for r in segment.residues])
        ref_seqs.append((chain_id, seq))

    mob_seqs = []
    for segment in mob_ca.segments:
        chain_id = segment.segid
        if chain_id == 'C':
            continue
        seq = ''.join([three2one[r.resname]for r in segment.residues])
        mob_seqs.append((chain_id, seq))

    for c, seq in mob_seqs:
        print(c, seq)
    for c, seq in ref_seqs:
        print(c, seq)
    # Perform Global pairwise alignment
    alignments = []
    # ali_indication = ''
    for (ref_chain_id, refseq), (mob_chain_id, mobseq) in zip(ref_seqs, mob_seqs):
        alignment = pairwise2.align.globalms(refseq, mobseq, 2, -1, -3, -2)
        alignment = alignment[0]
        alignment = pairwise2.format_alignment(*alignment).split('\n')
        alignments.append((ref_chain_id, mob_chain_id, alignment))
    # Concatenate the aligned sequences: all to 1 chain
    ref_ali = ''.join([x[2][0] for x in alignments])
    idc_ali = ''.join([x[2][1] for x in alignments])
    mob_ali = ''.join([x[2][2] for x in alignments])
    ref_ali, idc_ali, mob_ali

    # Create a list of residue indices
    counter = itertools.count()
    ref_ali_residx = [next(counter) if c != '-' else -1 for c in ref_ali]
    counter = itertools.count()
    mob_ali_residx = [next(counter) if c != '-' else -1 for c in mob_ali]

    # Matching and non-matching residue indices
    match_residx = []
    nonmatch_residx = []
    for c, x1, x2 in zip(idc_ali, ref_ali_residx, mob_ali_residx):
        if (c in [' ', '.']) or (-1 in [x1, x2]):
            nonmatch_residx.append((x1, x2))
        else:
            match_residx.append((x1, x2))

    # Matching residue indices
    match_residx_ref=[]
    match_residx_mob=[]
    i=0
    while i < len(match_residx):
        match_residx_ref.append(match_residx[i][0])
        match_residx_mob.append(match_residx[i][1])
        i +=1

    # Select atoms
    ref_sel = ref_ca[match_residx_ref]
    mob_sel = mob_ca[match_residx_mob]
    print(len(match_residx_ref))
    print(len(match_residx_mob))
    print(" ".join([str(i) for i in match_residx_ref]))
    print(" ".join([str(i) for i in match_residx_mob]))

    # Superimpose
    ref0 = ref_sel.positions - ref_sel.center_of_mass()
    mob0 = mob_sel.positions - mob_sel.center_of_mass()
    R, rmsd = align.rotation_matrix(mob0, ref0)
    print('RMSD after superimposition:', rmsd)

    mobile.atoms.translate(-mob_sel.center_of_mass())
    mobile.atoms.rotate(R)
    mobile.atoms.translate(ref_sel.center_of_mass())

    mobile.atoms.write(out)


mobile = '/mnt/nas_1/YangLab/loci/casE/data/output/CasE_14_P2V1/wt/wt_model_chain_swapBC_renum_ordered_seg.pdb'
reference = '/mnt/nas_1/YangLab/loci/casE/data/pdb/3BP7.pdb'
out= '/mnt/nas_1/YangLab/loci/casE/data/output/CasE_14_P2V1/wt/wt_model_to3BP7.pdb'
superimpose(mobile, reference, out)

A GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPREEPRAPWIEQEGPEYWDRNTQIYKAQAQTDRESLRNLRGYYNQSEAGSHTLQSMYGCDVGPDGRLLRGHDQYAYDGKDYIALNEDLRSWTAADTAAQITQRKWEAAREAEQRRAYLEGECVEWLRRYLENGKDKLERADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWEPSSQSTVPIV
B IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM
A GSHSMRYFHTSVSRPGRGEPRFITVGYVDDTLFVRFDSDAASPREEPRAPWIEQEGPEYWDRETQICKAKAQTDREDLRTLLRYYNQSEAGSHTLQNMYGCDVGPDGRLLRGYHQHAYDGKDYIALNEDLSSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKETLQRADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWEP
B MIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM
355
355
0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 8

/home/yang_loci/miniconda3/envs/enm_workshop/lib/python3.11/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/yang_loci/miniconda3/envs/enm_workshop/lib/python3.11/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


In [ ]:
0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 277 278 279 280 281 282 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375
0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375 376 377 378 379 380 381 382 383

0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 277 278 279 280 281 282 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375
0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375 376 377 378 379 380 381 382 383

molinfo list
set ref [atomselect 3 "residue 0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 277 278 279 280 281 282 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375 and name CA"]
set mob [atomselect 4 "residue 0 1 2 3 4 5 6 7 9 10 11 12 13 14 15 16 17 18 19 20 21 22 24 25 26 27 28 29 30 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 63 64 65 67 68 70 71 72 73 74 75 77 78 80 83 84 85 86 87 88 89 90 91 92 93 94 95 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 119 120 121 122 123 124 125 126 127 128 129 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 152 153 154 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 178 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371 372 373 374 375 376 377 378 379 380 381 382 383 and name CA"]
$ref num
$mob num
set M [measure fit $mob $ref]

set all [atomselect 4 "all"]
$all move $M

set rmsd [measure rmsd $mob $ref]
puts "RMSD after alignment = $rmsd"



In [1]:
import prody as pr

in_pdb = "/mnt/nas_1/YangLab/loci/casE/data/output/CasE_14_P2V1/wt/wt_model_chain.pdb"
out_pdb = "/mnt/nas_1/YangLab/loci/casE/data/output/CasE_14_P2V1/wt/wt_model_chain_swapBC_renum_ordered_seg.pdb"

p = pr.parsePDB(in_pdb)

# Original chains
chain_A = p.select("chain A").copy()
chain_B = p.select("chain B").copy()
chain_C = p.select("chain C").copy()

# Rename chain IDs and segment names
chain_A.setChids("A")
chain_A.setSegnames("A")

chain_C.setChids("B")      # original C -> new B
chain_C.setSegnames("B")

chain_B.setChids("C")      # original B -> new C
chain_B.setSegnames("C")

# Combine in desired output order: A, B, C
q = chain_A + chain_C + chain_B

# Renumber residues within each chain
for ch in ["A", "B", "C"]:
    sel = q.select(f"chain {ch}")

    old_resnums = sel.getResnums()
    mapping = {}
    new_resnums = []
    current = 1

    for r in old_resnums:
        if r not in mapping:
            mapping[r] = current
            current += 1
        new_resnums.append(mapping[r])

    sel.setResnums(new_resnums)

# Renumber atom serials
q.setSerials(range(1, q.numAtoms() + 1))

pr.writePDB(out_pdb, q)

print(f"Saved to: {out_pdb}")

/home/yang_loci/miniconda3/envs/tandem/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
@> 3232 atoms and 1 coordinate set(s) were parsed in 0.06s.


Saved to: /mnt/nas_1/YangLab/loci/casE/data/output/CasE_14_P2V1/wt/wt_model_chain_swapBC_renum_ordered_seg.pdb
